# VISU-laskentaesimerkki – Sydämen vajaatoiminnan ennustaminen kliinisestä datasta

Tämä on esimerkki siitä, kuinka omaa koodia voidaan ajaa VISU-laskentaympäristössä taulukkomuotoisen datan analysointiin.
Tärkeää on, että myös arkaluonteiseen dataan pääsee käsiksi vastaavalla tavalla siirtämällä data bind-liitettyyn kansioon VISUssa.
Tässä oppaassa käytetään avointa aineistoa sydämen vajaatoiminnan ennustamiseen kliinisistä mittauksista.

Käsitellyt aiheet:
- Datan lataaminen konttiin liitetystä kansiosta
- Koneoppimisluokittimien kouluttaminen binääriselle kohdemuuttujalle
- Taulukkomuotoisen datan käsittely
- Datan jakaminen toistettavien tulosten varmistamiseksi
- Binääriluokittimen tulosten arviointi
- Kuinka tulkita mallin päätöksiä?

Tämän oppaan aineisto perustuu Chiccon & Jurmanin tutkimukseen [1], jossa on kerätty kliiniset piirteet 299 sydämen vajaatoimintapotilaalta Allied Hospitalista Faisalabadissa (Punjab, Pakistan).

[1] Chicco, D., Jurman, G. Machine learning can predict survival of patients with heart failure from serum creatinine and ejection fraction alone. BMC Med Inform Decis Mak 20, 16 (2020). https://doi.org/10.1186/s12911-020-1023-5

---


# 1. Datan lataaminen ja esikäsittely

Jokaisen Python-koodin ensimmäinen vaihe on tarvittavien **kirjastojen** tuominen. Kirjastot sisältävät valmiita funktioita ja luokkia, joita voidaan käyttää sen sijaan, että jokainen funktio kirjoitettaisiin itse alusta asti. Esimerkiksi *numpy* mahdollistaa erilaisten matemaattisten operaatioiden tekemisen taulukoille, ja *scikit-learn* (sklearn) tarjoaa laajan valikoiman koneoppimisen työkaluja.

In [ ]:
# Vuorovaikutus käyttöjärjestelmän kanssa
import os

# Matemaattiset operaatiot
import numpy as np

# Informatiivisten kuvaajien luonti
import matplotlib.pyplot as plt

# Taulukkomuotoisen datan käsittely
import pandas as pd

# Scikit-learn on paketti, joka sisältää useita koneoppimisen työkaluja

# Esikäsittely
from sklearn.preprocessing import StandardScaler
# Aineiston jakaminen
from sklearn.model_selection import LeaveOneOut, KFold, StratifiedKFold, train_test_split
# Random forest -luokittelija
from sklearn.ensemble import RandomForestClassifier
# Esikäsittelyn ja luokittelun yhdistäminen prosessiksi
from sklearn.pipeline import Pipeline
# Tulosten arviointi
from sklearn.metrics import confusion_matrix, mean_squared_error, \
    accuracy_score, f1_score, precision_score, recall_score, \
    average_precision_score, precision_recall_curve, PrecisionRecallDisplay, roc_curve, RocCurveDisplay, auc, classification_report

# Muuttujien valinta ja tulkinta
import shap

Luetaan seuraavaksi data bind-liitetystä kansiosta (Dockerissa vastaavasti volyymista). Komennolla `data.info()` voimme tarkastella datan perusyhteenvetoa.

In [ ]:
# Muuttujat opastusta varten
seed = 42

# Ladataan sydämen vajaatoiminta-aineisto .csv-tiedostosta. Huomaa, että Dockerfilessa määriteltyä ympäristömuuttujaa käytetään tarkistamaan, ajetaanko muistikirjaa kontissa.
if os.environ.get("RUNNING_IN_CONTAINER") == "Yes":
    data_path = "/code/data/heart_failure_clinical_records_dataset.csv"
else:
    data_path = "data/heart_failure_clinical_records_dataset.csv"

data = pd.read_csv(data_path)
# Nopea kuvaus tiedoston sisällöstä
data.info()

Data näyttää koostuvan taulukosta, jossa on 13 saraketta ja jossa *DEATH_EVENT* on kohdemuuttuja. Osa muuttujista on kokonaislukuja, mutta joukossa on myös desimaalilukuja sisältäviä mittauksia, kuten *serum_creatinine* ja *platelets*. Alla on lyhyt kuvaus muuttujista.

Muuttuja | Kuvaus
--- | ---
**Age** | Potilaan ikä (vuotta)
**Aneamia**|Punasolujen tai hemoglobiinin väheneminen (totuusarvo)
**creatinine_phosphokinase**|CPK-entsyymin taso veressä (mcg/l)
**diabetes**|Onko potilaalla diabetes (totuusarvo)
**ejection_fraction**|Sydämestä poistuvan veren prosenttiosuus kussakin supistuksessa
**high_blood_pressure**|Onko potilaalla verenpainetauti (totuusarvo)
**platelets**|Verihiutaleet veressä (tuhatta hiutaletta/ml)
**serum_creatinine**|Seerumin kreatiniinitaso veressä (mg/dl)
**serum_sodium**|Seerumin natriumtaso veressä (mEq/l)
**sex**|Mies vai nainen (binäärinen, nainen=0, mies=1)
**smoking**|Tupakoiko potilas (totuusarvo)
**time**|Seuranta-aika (päivää)
**DEATH_EVENT**|Menehtyikö potilas seuranta-aikana (totuusarvo)


In [ ]:
# Kunkin muuttujan tilastolliset tunnusluvut
data.describe()

In [ ]:
# Kuinka moni potilas menehtyi sydämen vajaatoimintaan?
positives = np.sum(data['DEATH_EVENT'])
negatives = len(data['DEATH_EVENT']) - positives
print(f'Aineisto sisältää {positives} positiivista ja {negatives} negatiivista tapausta.')

# 2. Datan jakaminen

Nyt kun olemme silmäilleet aineistoa, voidaan alkaa pohtia, mitkä **muuttujat** ovat tärkeitä.

- **K1: Onko syötemuuttujien joukossa sellaisia, jotka eivät ole kliinisesti merkityksellisiä tai mielekkäitä?**

Data tulisi jakaa **opetus- ja testijoukkoon**. Testijoukkoa tulisi käyttää vasta tutkimuksen lopussa, kun kaikki parametrit ja menetelmät on lyöty lukkoon. Tämä tarkoittaa, ettei koneoppimisputkea tulisi enää muuttaa sen jälkeen, kun testidatan lopulliset tulokset on nähty.

**Kiinteän satunnaissiemenen** avulla voimme ottaa aineistosta satunnaisotoksen, mutta varmistaa, että sama tulos voidaan toistaa myöhemmin.

<details>
    <summary>Katso vastaus kysymykseen K1 tästä</summary>
    VASTAUS K1:een: Time-muuttuja kuvaa seuranta-aikaa ja korreloi ilmeisesti voimakkaasti kuoleman kanssa (pitkä seuranta-aika johtaa todennäköisemmin kuolemaan). Sitä ei tulisi käyttää, koska se tarjoaa vain vähän diagnostista arvoa.
</details>

In [ ]:
# Satunnaissiemen toistettavuuden varmistamiseksi
seed = 1
# Jaetaan opetus- ja testidata (67/33-suhteessa) asettamalla potilaan menehtyminen kohdemuuttujaksi
data_train, data_test, Y_train, Y_test = train_test_split(data.iloc[:, :-1], data['DEATH_EVENT'].values, test_size=0.33, random_state=seed, shuffle=True)

# 3. Muuttujien valinta ja skaalaus

Seuraavaksi valmistelemme kaksi eri syöteaineistoa: ensimmäisen, jossa on 11 ensimmäistä muuttujaa (ilman seuranta-aikaa), ja toisen, jossa on vain kaksi muuttujaa. Koneoppimisessa muuttujat on usein tärkeää skaalata **vähentämällä keskiarvo ja jakamalla keskihajonnalla** (StandardScaler), jotta eri yksiköissä olevat muuttujat (esim. ikä vuosina vs. laboratoriotulokset) olisivat keskenään vertailukelpoisia.

Emme skaalaa koko opetusaineistoa valmiiksi tässä vaiheessa. Jos skaalaisimme datan nyt, ristiinvalidoinnin (K-Fold) sisällä olevat validointiaineistot "vuotaisivat" tietoa opetusvaiheeseen.
Sen sijaan jätämme datan raakamuotoon, ja varsinainen skaalaus tapahtuu automaattisesti opetusvaiheessa Scikit-Learnin Pipeline-prosessina. Tällöin testi- ja validointiaineistoa skaalataan vain opetusdatasta opitulla keskiarvolla ja -hajonnalla, mikä kuvaa hyvin todellisuutta uuden datan kanssa.

Kohdemuuttuja kertoo, kuoliko potilas sydämen vajaatoimintaan. Skaalaamme vain syötemuuttujat, emme kohdemuuttujaa, joka sisältää vain luokkatiedot 0 ja 1.

Katsotaan miltä data näyttää ennen ja jälkeen skaalauksen, säästetään kuitenkin varsinainen skaalaus vasta opetussilmukkaan.

In [ ]:
# 1. aineisto: 11 muuttujaa
X_train = np.array(data_train.iloc[:, :11])
X_test = np.array(data_test.iloc[:, :11])

# 2. aineisto: 2 muuttujaa
features_2f = ['ejection_fraction', 'serum_creatinine']
X2_train = np.array(data_train[features_2f])
X2_test = np.array(data_test[features_2f])

# Skaalauksen vaikutus

print('Ennen skaalausta:')
print(data_train[features_2f].head(3))

# Esimerkki aineiston skaalauksesta
demo_scaler = StandardScaler()
X2_train_demo_scaled = demo_scaler.fit_transform(data_train[features_2f])

print('\nSkaalauksen jälkeen (keskiarvo ~0, keskihajonta ~1):')
print(pd.DataFrame(X2_train_demo_scaled, columns=features_2f).head(3))

# 4. Mallin kouluttaminen

Koneoppimismallin kouluttamista varten voimme kirjoittaa funktion, joka suorittaa mallin koulutuksen (**koulutussilmukka**). Funktion sisällä suoritamme **ristiinvalidoinnin**, jossa data jaetaan useisiin opetus- ja validointi**osiin** (folds). Ristiinvalidointimenettely on havainnollistettu alla (sininen kuvaa validointidataa, lähde: [Scikit-learn](https://scikit-learn.org/stable/modules/cross_validation.html)).

Kuten ensimmäisessä osiossa huomasimme, aineistossa on enemmän negatiivisia (selvinneitä) kuin positiivisia (sydämen vajaatoiminta) tapauksia. Datan **ositus (stratifiointi)** on hyödyllistä, jotta positiivisten ja negatiivisten tapausten jakauma säilyy opetus- ja validointitaitoksissa. Tämä voidaan toteuttaa **StratifiedKFold**-luokalla.

Ennustajana käytämme **Random forest -luokittelijaa**. Se koostuu useista päätöspuista, jotka lopulta yhdistyvät yhdeksi binääriseksi ennusteeksi. Luokitin koulutetaan usein opetusdatan osajoukoilla, jotka poimitaan uudelleenotannalla (**bootstrap-otanta**). Bootstrap-otanta on oletuksena käytössä.

In [ ]:
plt.figure(figsize=(15,15))
img=plt.imread('/code/tutorial/cross_validation.png')
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Koulutussilmukka
def train_skfold(feature, target, metric=accuracy_score, splits=5, threshold=0.5, seed=42):
    """
    Kouluttaa Random forest -luokittimen käyttäen ositettua K-kertaista ristiinvalidointia (stratified K-fold).

    :param feature: Mallin syöte
    :param target: Mallin oikea vastaus (ground truth)
    :param metric: Mittari, jolla ennusteita ja oikeita vastauksia verrataan
    :param splits: Ristiinvalidoinnin jakojen määrä
    :param threshold: Binääriennusteen kynnysarvo
    :return: Kaikkien osien yhdistetyt ennusteet, oikeat vastaukset ja mittarit
    """

    # Alustetaan SKFold-jako, sekoitetaan data ennen jakamista
    skfold = StratifiedKFold(n_splits=splits, shuffle=True, random_state=seed)

    # Alustetaan listat
    scores, predictions, targets, models = [], [], [], []

    # Muutetaan Pandas-data NumPy-muotoon, jotta numeerinen indeksointi toimii luotettavasti
    feature = feature.to_numpy() if hasattr(feature, 'to_numpy') else np.array(feature)
    target = target.to_numpy() if hasattr(target, 'to_numpy') else np.array(target)

    # Ristiinvalidoinnin ositus
    for train_idx, test_idx in skfold.split(feature, target):
      features_train = feature[train_idx]
      features_val = feature[test_idx]
      target_train = target[train_idx]
      target_val = target[test_idx]

      # Yhdistetään esikäsittely ja malli prosessiksi
      model = Pipeline([
            ('scaler', StandardScaler()),  # Vähennetään opetusaineiston keskiarvo ja jaetaan keskihajonta
            ('classifier', RandomForestClassifier(random_state=seed))  # Käytetään mallina Random forest -luokittelijaa oletusparametreilla
        ])

      # Sovitetaan malli opetusdataan
      model.fit(features_train, target_train)

      # Ennustetaan validointijoukolle (positiivisen luokan 1 todennäköisyys)
      prediction = model.predict_proba(features_val)[:, 1]

      # Arvioidaan suorituskyky
      score = metric(prediction > threshold, target_val)

      # Lisätään listoihin
      scores.append(score)
      predictions.extend(prediction)
      targets.extend(target_val)
      models.append(model)

    # Muunnetaan tulokset taulukoiksi
    predictions = np.array(predictions)
    targets = np.array(targets)

    return predictions, targets, scores, models



Nyt kun koulutussilmukka on määritelty, tarvitsee vain kutsua sitä opetusdatalla ja muilla koulutusparametreilla.

In [ ]:
# Koulutetaan Random Forest -luokittelija ositetulla k-kertaisella ristiinvalidoinnilla
predictions, targets, accuracies, models = train_skfold(X_train, Y_train, metric=accuracy_score, splits=5, seed=seed)

# Tulos validointidatalla
print('Random Forestin tarkkuus kullekin ristiinvalidoinnin osalle: ', accuracies)
accuracy_avg = np.mean(accuracies) * 100
print(f'\nKaikkien osien keskimääräinen tarkkuus: {accuracy_avg:.1f}%')

# 5. Tulosten analysointi

Koulutettu malli saavutti keskimäärin 75,5 %:n **validointitarkkuuden**. Huomaa, että koska jaoimme datan viisi kertaa, koulutimme käytännössä viisi erillistä mallia. Joidenkin mallien suorituskyky oli keskiarvoa parempi: tämä voi tarkoittaa, että malli oppi paremman parametrijoukon päätöspuille, tai että joukossa on "helpompia" luokiteltavia näytteitä (kuten poikkeuksellisen korkea verenpaine).

Seuraavaksi voimme analysoida tuloksia tarkemmin testijoukolla. **Huomaa, että oikeissa kokeissa testijoukkoa tulisi arvioida vasta, kun kaikki mallin parametrit ja suunnitteluvalinnat on lyöty lukkoon!** Muuten tulosten yliarvioinnin riski kasvaa. Koneoppimistutkijat ovat usein raportoineet korkeita suorituskykylukemia tutkimusdatalla, mutta huomanneet myöhemmin, ettei menetelmä toimi käytännössä **ylisovittumisen (overfitting)** vuoksi.

Käytimme suorituskykymittarina myös tarkkuutta (accuracy). Se on suosittu mittari, mutta sitä tulisi välttää, kun aineisto on **vinoutunut (engl. biased)**, eli luokat eivät ole yhtä suuressa määrin edustettuina. Koska tarkkuus on yksinkertaisesti oikeiden ennusteiden osuus, pienempi luokka saattaa jäädä analyysissa vaille huomiota.

Ennusteiden tarkempaa analysointia varten voimme piirtää sekaannusmatriisin (confusion matrix, selitetty alla).

In [ ]:
plt.figure(figsize=(15,15))
img=plt.imread('/code/tutorial/confusion_matrix.png')
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Lasketaan sekaannusmatriisi validointidatalla
conf = confusion_matrix(targets, predictions > 0.5)
# Näytetään tulokset taulukkona, oikeat ennusteet sijaitsevat diagonaalilla
print(pd.DataFrame(conf, index=['Todellinen negatiivinen', 'Todellinen positiivinen'],
             columns=['Ennustettu negatiivinen', 'Ennustettu positiivinen']))

Sekaannusmatriisin perusteella näemme, että korkeasta tarkkuudesta huolimatta puolet positiivisista tapauksista luokitellaan väärin.

In [ ]:
# Ennustetaan testidata kaikkien ristiinvalidointitaitosten keskiarvona

# Ennuste testijoukolle
predictions_test = []
for model in models:
    prediction = model.predict_proba(np.array(X_test))[:, 1]
    predictions_test.append(prediction)
# Keskiarvoistetaan ennusteet ja binarisoidaan tulos
predictions_test = np.mean(predictions_test, axis=0)

# Todennäköisyyden kynnysarvo
th = 0.5

# Lasketaan sekaannusmatriisi testidatalla
conf = confusion_matrix(Y_test, predictions_test > th)
# Näytetään tulokset taulukkona, oikeat ennusteet sijaitsevat diagonaalilla
print(pd.DataFrame(conf, index=['Todellinen negatiivinen', 'Todellinen positiivinen'],
             columns=['Ennustettu negatiivinen', 'Ennustettu positiivinen']))

# Tulokset testijoukolle
print(classification_report(Y_test, predictions_test > th), target_names=['Negatiivinen', 'Positiivinen'])

Testidatalla alle puolet positiivisista tapauksista luokitellaan oikein.

Positiivisessa luokassa saavutimme suhteellisen korkean täsmällisyyden (precision) 0.59, mutta matalan herkkyyden (recall) 0.45. Tämä tarkoittaa, että potilaat, jotka ennustetaan positiivisiksi, kuolevat todennäköisesti sydämen vajaatoimintaan. Toisaalta matala herkkyys tarkoittaa, että moni potilas, joka tulee kuolemaan, ennustetaan negatiiviseksi (terveeksi).

Huomaa, että edellisissä mittareissa mallin tulos jouduttiin **binarisoimaan** (tässä tapauksessa 0,5 **kynnysarvolla**). Ennusteiden tarkempaa analysointia varten voimme piirtää kuvaajia, jotka kattavat koko kynnysarvojen välin 0:sta 1:een.

# 6. ROC- ja PR-käyrien analysointi

Mallin ominaisuuksien tarkempaa analysointia varten voimme piirtää **ROC-käyrän (Receiver Operating Characteristic) ja täsmällisyys–herkkyys-käyrän (precision-recall curve, PRC)**. Käyrät muodostetaan muuttamalla **ennustekynnystä** ja piirtämällä annetut mittarit x- ja y-akseleille.

ROC-käyrissä piirretään **todellinen positiivisuusaste (true positive rate)** ja väärä positiivisuusaste (false positive rate). PRC-käyrässä precision ja recall piirretään vastaavalla tavalla. Koska täsmällisyys ja herkkyys ottavat luokkien epätasapainon paremmin huomioon, se tulisi aina piirtää ROC-käyrän rinnalla vinoutuneella aineistolla (lisätietoja: [Saito & Rehmsmeier](https://doi.org/10.1371/journal.pone.0118432)).

Molemmissa tapauksissa satunnainen arvaus piirretään pisteviivalla, kun taas täydellisen luokittelijan ennusteet näkyisivät kuvaajassa punaisella.

In [ ]:
# ROC-käyrä

fpr, tpr, _ = roc_curve(Y_test, predictions_test)
# Lasketaan käyrän alle jäävä pinta-ala (AUC)
auc_score = auc(fpr, tpr)
RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc_score, name='Random Forest').plot()
# Lisätään perustaso (katkoviiva)
plt.plot([0, 1], [0, 1], '--', color='black')
# Lisätään optimiviiva (punainen)
plt.plot([0, 0, 1], [0, 1, 1], '-', alpha=0.5, color='red')
plt.show()

# PRC-käyrä

# Lasketaan satunnaisarvauksen perustaso luokkajakauman avulla
positives = np.sum(Y_train)
negatives = len(Y_train) - positives
baseline = positives / (positives + negatives)

# Lasketaan PR-käyrä ja keskimääräinen precision
prec, rec, _ = precision_recall_curve(Y_test, predictions_test)
ap = average_precision_score(Y_test, predictions_test)
# Näytetään PRC ja perustaso
pr_plot = PrecisionRecallDisplay(precision=prec, recall=rec, average_precision=ap, name='Random Forest').plot()
plt.plot([0, 1], [baseline, baseline], '--',
         color='black', alpha=0.5, linewidth=2)
# Lisätään optimiviiva (punainen)
plt.plot([0, 1, 1], [1, 1, 0], '-', alpha=0.5, color='red')
plt.ylim([0, 1.01])
plt.show()
print(f'Perustason precision = {baseline*100}%')

# 7. Muuttujien tärkeys

Nykyinen Random forest -luokitin koulutettiin käyttäen 11:tä eri parametria (**muuttujaa**). Mallin päätösten ymmärtämiseksi olisi hyödyllistä tietää, mitkä muuttujat ovat tärkeimpiä.

SHAP (SHapley Additive exPlanations) käyttää peliteoreettista lähestymistapaa parametrien tärkeyden arviointiin. Lloyd Shapley voitti taloustieteen Nobel-palkinnon Shapley-arvojen keksimisestä. Koneoppimisessa teoriaa voidaan käyttää laskemaan, mitkä muuttujat vaikuttavat eniten tiettyyn ennusteeseen.

In [ ]:
# Tulos validointidatalla
print('Random Forestin tarkkuus ristiinvalidoinnin osille: ', accuracies)

Ristiinvalidoinnin osan 3 malli näytti antavan parhaan tarkkuuden viidestä taitoksesta (0,875). Katsotaan, mitkä muuttujat ovat tärkeitä juuri tämän mallin ennusteille.

In [ ]:
# Tulkitaan ristiinvalidoinnin 1. osamallin ennusteita SHAP-menetelmällä: SHapley Additive exPlanations
cv_fold = 0

# SHAP ei tue Pipelinea, tehdään skaalaus erikseen
X_test_scaled = models[cv_fold]['scaler'].transform(X_test)

# Lasketaan SHAP-arvot
explainer = shap.TreeExplainer(models[cv_fold]['classifier'])  # Valitaan Pipelinesta luokittelija
shap_values = explainer.shap_values(X_test_scaled)

# Visualisoidaan selitykset luokalle 1 (potilaan kuolema)
shap.initjs()  # Alustetaan Javascript
shap.summary_plot(shap_values[:,:,1], X_test_scaled, feature_names=data_train.columns[:11])

Yhteenvetokuvaajassa punainen väri edustaa kyseisen muuttujan korkeaa arvoa, kun taas oikealla puolella oleva piste tarkoittaa suurempaa todennäköisyyttä tulla ennustetuksi positiiviseksi. Muuttujat on järjestetty niiden suhteellisen tärkeyden mukaan.

Voimme huomata, että korkea serum_creatinine-arvo (punainen) johtaa korkeaan kuolemanriskiin (tämän mallin ennusteen perusteella). Sama pätee matalaan ejection_fraction-arvoon (sininen).

Katsotaan seuraavaksi yksittäistä potilasta.

In [ ]:
patient_ID = 40

shap.initjs()
# Yhden potilaan ennuste
print('Ennustettu kuolema: ', models[cv_fold].predict(X_test_scaled[patient_ID].reshape(1, -1)), ', todellinen luokka: ', Y_test[patient_ID])

# Force plot yksittäiselle potilaalle
shap.force_plot(explainer.expected_value[1], # Perustaso (baseline)
                shap_values[patient_ID, :, 1].reshape(1, -1), # Shap-arvot (vaikutus ennusteeseen)
                models[cv_fold]['scaler'].inverse_transform(X_test_scaled[patient_ID].reshape(1, -1)), # Muuttujan arvo, käytetään StandardScaleria poistamaan standardointi
                feature_names=data_train.columns[:11]) # Muuttujan nimi

Tämä potilas ennustettiin oikein positiiviseksi. Korkea seerumin kreatiinipitoisuus ja matala ejektiofraktio vaikuttivat ennusteeseen. Arvot 0,79 ja seuraavan kuvaajan 0,19 ovat mallin ennusteita. Huomaa, että force plot -kuvaajan punainen väri viittaa muuttujiin, jotka vaikuttavat positiivisen ennusteen suuntaan.

Huomaa myös, että muuttujien arvot on skaalattu käänteisesti alkuperäisten arvojen näyttämiseksi.

In [ ]:
patient_ID = 4

shap.initjs()
print('Ennustettu kuolema: ', models[cv_fold].predict(X_test[patient_ID].reshape(1, -1)), ', todellinen luokka: ', Y_test[patient_ID])

# Force plot yksittäiselle potilaalle
shap.force_plot(explainer.expected_value[1], # Perustaso (baseline)
                shap_values[patient_ID, :, 1].reshape(1, -1), # Shap-arvot (vaikutus ennusteeseen)
                models[cv_fold]['scaler'].inverse_transform(X_test_scaled[patient_ID].reshape(1, -1)), # Muuttujan arvo
                feature_names=data_train.columns[:11]) # Muuttujan nimi

Tässä on esimerkki potilaasta, jonka ennustettiin selvästi selviävän. Huomaa matala seerumin kreatiini ja korkea ejektiofraktio. Shap-tulos on perustason kynnysarvon 0.5 alapuolella, ja ennuste on negatiivinen.

Useita force plot -kuvaajia voidaan kääntää ja pinota yhteen useamman potilaan tarkastelemiseksi:

In [ ]:
shap.initjs()  # Alustetaan Javascript
# Force plot koko testijoukolle
shap.force_plot(explainer.expected_value[1],
                shap_values[:,:,1],
                models[cv_fold]['scaler'].inverse_transform(X_test_scaled),
                feature_names=data_train.columns[:11])

**K2: Tarkastele yllä olevia SHAP-tulkintakuvaajia. Mitkä muuttujat ovat tärkeimpiä sydämen vajaatoiminnan ennustamisessa?**
<details>
    <summary>Katso vastaus kysymykseen K2 tästä</summary>
    VASTAUS K2:een: Nykyinen malli käyttää kaikkia 11 käyttökelpoista kliinistä muuttujaa. Chiccon & Jurmanin tutkimus kuitenkin viittaa siihen, että parhaat muuttujat sydämen vajaatoiminnan arviointiin ovat seerumin kreatiniini ja ejection fraction. On tärkeää huomata, että sama johtopäätös voidaan tehdä yllä olevasta pystysuuntaisesta yhteenvetokuvaajasta. Voimme lopuksi testata, antaisiko vain näiden kahden muuttujan käyttö hyväksyttäviä tuloksia. Menetelmää olisi loppujen lopuksi helpompi käyttää, jos potilaasta tarvitsee kerätä vähemmän mittauksia tai tietoa. Mallilla, jossa on vähemmän muuttujia, on myös pienempi riski ylisovittua.
</details>

In [ ]:
# Koulutetaan Random Forest -luokitin ositetulla k-kertaisella ristiinvalidoinnilla
predictions_2f, targets_2f, accuracies_2f, models_2f = train_skfold(X2_train, Y_train, metric=accuracy_score, splits=5, seed=seed)

# Tulos validointidatalla
print('Random Forestin tarkkuus kullekin ristiinvalidoinnin taitokselle: ', accuracies_2f)
accuracy_avg = np.mean(accuracies_2f) * 100
print(f'\nKeskimääräinen tarkkuus taitosten yli: {accuracy_avg:.1f}%')

Voimme nyt verrata tuloksia 11 muuttujaa käyttävän mallin kanssa.

In [ ]:
# Ennustetaan testidata kaikkien ristiinvalidointitaitosten keskiarvona

# Ennuste testijoukolle
predictions_test_2f = []
for model in models_2f:
    prediction = model.predict_proba(X2_test)[:, 1]
    predictions_test_2f.append(prediction)

# Keskiarvoistetaan ennusteet ja binarisoidaan tulos
predictions_test_2f = np.mean(predictions_test_2f, axis=0)

# Todennäköisyyden kynnysarvo
th = 0.5

# Lasketaan sekaannusmatriisi testidatalla
conf = confusion_matrix(Y_test, predictions_test_2f > th)
# Näytetään tulokset taulukkona, oikeat ennusteet sijaitsevat diagonaalilla
print(pd.DataFrame(conf, index=['Todellinen negatiivinen', 'Todellinen positiivinen'],
             columns=['Ennustettu negatiivinen', 'Ennustettu positiivinen']))

print('\n11 muuttujaa:')
print(classification_report(Y_test, predictions_test > th, target_names=['Negatiivinen', 'Positiivinen']))

print('\n2 muuttujaa:')
print(classification_report(Y_test, predictions_test_2f > th, target_names=['Negatiivinen', 'Positiivinen']))

# Yhteenveto

Tulosten perusteella voimme tehdä muutamia tärkeitä havaintoja:

- Tärkeimmät muuttujat olivat `serum_creatinine` ja `ejection_fraction`, samankaltaisia havaintoja raportoivat myös Chicco & Jurman
- Vain kahden muuttujan käyttö antaa testijoukolla paremman suorituskyvyn kuin 11 muuttujan malli.
- Validointitulokset olivat aina parempia kuin testitulokset. Tämä havainnollistaa **ylisovittumisen (overfitting)** vaikutusta optimiratkaisua etsittäessä

Opit myös käyttämään **ristiinvalidointia** datan käytön maksimoimiseksi. Tämä mahdollistaa tehokkaan mallin koulutuksen, kun opetusdatan määrä on rajallinen. Muita tärkeitä aiheita olivat **ositus (stratifiointi)** (luokkajakaumien säilyttäminen) ja **bootstrap-otanta** (osanäytteiden poiminta takaisinpanolla).

Koska koneoppimismalleja käsitellään usein mustana laatikkona, on tärkeää oppia tekniikoita, jotka lisäävät mallin **tulkittavuutta**. SHAP-analyysi on erinomainen esimerkki **muuttujien tärkeyden** visualisoinnista. Lisätietoa SHAP-analyysistä löydät [SHAP Python -paketista](https://github.com/slundberg/shap) sekä [Lundbergin ja muiden artikkelista](https://www.nature.com/articles/s41551-018-0304-0.epdf?author_access_token=vSPt7ryUfdSCv4qcyeEuCdRgN0jAjWel9jnR3ZoTv0PdqacSN9qNY_fC0jWkIQUd0L2zaj3bbIQEdrTqCczGWv2brU5rTJPxyss1N4yTIHpnSv5_nBVJoUbvejyvvjrGTb2odwWKT2Bfvl0ExQKhZw%3D%3D)